# Device Placement and Vectorization — Visual Intuition Edition

We are not just going to time two patterns.

We are going to **build small visualizers** that draw the hidden cost of moving data back and forth, so you can *see* why the rule "move once, compute many times" is not a micro-optimization — it is the difference between a GPU that feels broken and one that feels magical.

Inspired by the diagnostic visualization style in high-signal teaching notebooks (Karpathy's makemore lectures especially): the pictures do the teaching.


## 1. We build the visual language together

The two ideas we need to see:
- A simplified memory hierarchy (CPU RAM ↔ PCIe bus ↔ GPU VRAM)
- The "transfer storm" that happens when you touch data across the bus on every iteration


In [ ]:
import time
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
%matplotlib inline

plt.rcParams["figure.dpi"] = 140
plt.rcParams["font.size"] = 10


In [ ]:
def draw_memory_hierarchy(pcie_cost=8.0, title="Memory Hierarchy (simplified)"):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)

    # CPU box
    cpu = FancyBboxPatch((0.5, 2), 3, 2, boxstyle="round,pad=0.1", 
                         facecolor="#e3f2fd", edgecolor="#1565c0", linewidth=2)
    ax.add_patch(cpu)
    ax.text(2, 3, "CPU RAM\n(Host)", ha="center", va="center", fontsize=11, fontweight="bold")

    # PCIe pipe (width encodes cost)
    pipe_width = min(1.8, max(0.4, pcie_cost / 5))
    ax.annotate("", xy=(6.5, 3), xytext=(3.8, 3),
                arrowprops=dict(arrowstyle="->", color="#d32f2f", lw=pipe_width*2))
    ax.text(5.1, 3.6, f"PCIe bus\n~{pcie_cost:.1f}× slower", ha="center", va="bottom", 
            fontsize=9, color="#d32f2f", fontweight="bold")

    # GPU box
    gpu = FancyBboxPatch((7, 1.5), 4.5, 3, boxstyle="round,pad=0.1",
                         facecolor="#e8f5e9", edgecolor="#2e7d32", linewidth=2)
    ax.add_patch(gpu)
    ax.text(9.25, 3.5, "GPU", ha="center", va="center", fontsize=11, fontweight="bold", color="#1b5e20")
    ax.text(9.25, 2.8, "VRAM + thousands of cores", ha="center", va="center", fontsize=9)

    ax.text(6, 0.6, "Red thick = high repeated transfer cost. Blue thin = one-time good transfer.", 
            ha="center", fontsize=9, style="italic", color="#555")
    return fig


In [ ]:
def draw_transfer_storm(n_copies=20000, title=None):
    fig, ax = plt.subplots(figsize=(13, 4.5))
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 5)
    ax.axis("off")
    if title is None:
        title = f"Transfer Storm — {n_copies:,} individual crossings (the bad pattern)"
    ax.set_title(title, fontsize=13, fontweight="bold", color="#b71c1c", pad=10)

    # CPU side (left)
    ax.add_patch(FancyBboxPatch((0.3, 1.5), 2.5, 2, boxstyle="round,pad=0.05",
                                facecolor="#ffcdd2", edgecolor="#c62828", lw=1.5))
    ax.text(1.55, 2.5, "CPU", ha="center", va="center", fontsize=10, fontweight="bold")

    # GPU side (right)
    ax.add_patch(FancyBboxPatch((10, 1.5), 2.5, 2, boxstyle="round,pad=0.05",
                                facecolor="#c8e6c9", edgecolor="#2e7d32", lw=1.5))
    ax.text(11.25, 2.5, "GPU", ha="center", va="center", fontsize=10, fontweight="bold")

    # Draw a sample of the storm (we can't draw 20k arrows visually)
    n_show = min(18, max(6, n_copies // 1200))
    for i in range(n_show):
        y = 1.7 + (i / max(1, n_show-1)) * 1.6
        ax.annotate("", xy=(9.7, y), xytext=(3.1, y),
                    arrowprops=dict(arrowstyle="->", color="#d32f2f", lw=1.2, alpha=0.65))
    
    ax.text(6.5, 4.2, f"≈ {n_copies:,} tiny round-trips across the bus", 
            ha="center", fontsize=11, color="#b71c1c", fontweight="bold")
    ax.text(6.5, 0.6, "Every arrow you see (and thousands more) costs real time and energy.
"
            "This is what a Python-level elementwise loop on device-resident data can feel like
"
            "if you are also moving things implicitly, or the mental model of repeated host-device chatter.",
            ha="center", fontsize=9, style="italic")
    return fig


In [ ]:
def draw_clean_placement(title="Clean Placement — one transfer, all compute local"):
    fig, ax = plt.subplots(figsize=(13, 4.5))
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 5)
    ax.axis("off")
    ax.set_title(title, fontsize=13, fontweight="bold", color="#1b5e20", pad=10)

    # CPU
    ax.add_patch(FancyBboxPatch((0.3, 1.5), 2.5, 2, boxstyle="round,pad=0.05",
                                facecolor="#e3f2fd", edgecolor="#1565c0", lw=1.5))
    ax.text(1.55, 2.5, "CPU", ha="center", va="center", fontsize=10, fontweight="bold")

    # One good thick blue arrow (the single upfront move)
    ax.annotate("", xy=(9.7, 2.5), xytext=(3.1, 2.5),
                arrowprops=dict(arrowstyle="->", color="#1565c0", lw=6))
    ax.text(6.5, 3.3, "ONE transfer (the only time we cross the bus)", 
            ha="center", fontsize=10, color="#1565c0", fontweight="bold")

    # GPU box — now full of happy compute
    ax.add_patch(FancyBboxPatch((10, 1.2), 2.5, 2.6, boxstyle="round,pad=0.05",
                                facecolor="#c8e6c9", edgecolor="#2e7d32", lw=2))
    ax.text(11.25, 3.2, "GPU", ha="center", va="center", fontsize=10, fontweight="bold")
    ax.text(11.25, 2.4, "All compute", ha="center", va="center", fontsize=9)
    ax.text(11.25, 1.9, "happens here", ha="center", va="center", fontsize=9, style="italic")

    ax.text(6.5, 0.5, "The vectorized operation (vals * 2) happens entirely on the GPU.
"
            "Zero per-element chatter. This is the picture you want in your head for the rule.",
            ha="center", fontsize=9, style="italic")
    return fig


Run the three visual builders we just defined. These are the "hero diagrams" for the lesson.


In [ ]:
draw_memory_hierarchy(pcie_cost=8.0)
plt.show()


In [ ]:
draw_transfer_storm(n_copies=20000)
plt.show()


In [ ]:
draw_clean_placement()
plt.show()


## 2. The actual experiment (the numbers that feed the pictures)

Now we run the real micro-benchmarks. The timings we capture will later be used to label and calibrate the visuals.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

vals = torch.randn(20000, device=device)
out = torch.zeros_like(vals)

# Make timings fair: synchronize before and after each measurement.
if device == "cuda":
    torch.cuda.synchronize()

In [ ]:
# The "bad" pattern — explicit Python loop (even on device tensor)
t0 = time.perf_counter()
for i in range(vals.numel()):
    out[i] = vals[i] * 2.0
if device == "cuda":
    torch.cuda.synchronize()
loop_time = time.perf_counter() - t0
print("Python loop time:", round(loop_time, 4))

In [ ]:
# The good pattern — vectorized, stays on device
t1 = time.perf_counter()
vec = vals * 2.0
if device == "cuda":
    torch.cuda.synchronize()
vec_time = time.perf_counter() - t1
print("Vectorized time :", round(vec_time, 4))
print("Speedup         :", round(loop_time / max(vec_time, 1e-9), 1), "×")

Look at the two pictures you generated earlier.

The red storm diagram is the visual explanation for why the Python loop was slow even though the tensor "lived" on the GPU: the loop creates many tiny Python-level steps and repeated device work, so the program spends more time coordinating and moving through the pipeline than doing the actual math.

The clean blue arrow diagram is what the vectorized line does under the hood: one large GPU kernel path with the heavy work staying local to the device. That is the mental model you want when you review training or ETL code in production.

Now let's make the pictures even more personal.

## 3. Make the visual your own (the soul moment)

Before you change the size of the storm, here is the practical rule to keep in mind:
- If you see `.item()`, `.cpu()`, or a Python loop in a hot path, you are usually creating the expensive storm.
- If you move the data once and then do the heavy work on the GPU, you are following the single blue arrow.

Change the number of elements and re-draw the storm. Feel how the visual density of pain scales with the size of the "loop" you wrote.

In [ ]:
# Play with this number
n = 35000
draw_transfer_storm(n_copies=n, title=f"What your {n:,}-element Python loop looks like in the bus")
plt.show()


Now draw the memory hierarchy with a higher "PCIe cost" multiplier (imagine a slower link or more contention).


In [ ]:
draw_memory_hierarchy(pcie_cost=25.0, 
    title="Same hierarchy on a contended or older link — every red crossing hurts more")
plt.show()

# Tiny placement rule of thumb:
# bad  -> move small pieces back and forth inside a Python loop
# good -> move once, then let the GPU do the heavy compute in one vectorized pass
sample = vals[:8].cpu()
bad_example = [float(x) * 2.0 for x in sample]
good_example = vals[:8] * 2.0
print("Bad example (CPU-side Python):", bad_example[:3])
print("Good example (GPU-side vectorized):", good_example[:3].cpu().tolist())

## Checkpoint — What did the pictures teach you that pure text could not?

**Question:** When you changed `n` from 20k to 35k and re-drew the storm, what changed in the picture, and what does that tell you about real production code that accidentally does per-row or per-token host-device chatter inside a hot loop?

(Write one sentence. Then look at the clean placement picture again. That is the mental model you want when you review someone else's training or ETL code.)

If the red storm still feels abstract, go back and re-run the vectorized timing cell, then immediately call `draw_clean_placement()` again. The contrast is the teaching.


## Lesson Recap — What you now carry in your head

- You have two concrete pictures: the red transfer storm (the hidden tax) and the single clean blue arrow (the desired state).
- You saw that the "slow" pattern is not always about arithmetic — it is about how many times data crosses the bus and how much Python-level chatter surrounds each tiny operation.
- You can now look at a piece of GPU code and ask: "How many times is data moving, and can I draw it as a storm or as a single arrow?"
- The visualizers you just used and mutated are now part of your debugging toolkit.


## Role Lens

**DevOps / MLOps:** When a training job shows low GPU utilization but high PCIe traffic in nvidia-smi or DCGM, you now have the exact mental image (the red storm) to explain to the team why "we just moved the model to GPUs" did not produce the expected speedup.

**Data Science:** Feature engineering that produces CPU tensors inside the training loop, or logging that forces sync points on every batch, looks exactly like the storm diagram. You now have a picture to put in the PR comment.

**Data Engineering:** Every time you write a RAPIDS or Polars-GPU pipeline, you are making the same placement decision at larger scale. The single blue arrow is the difference between a pipeline that finishes before the next data lands and one that is permanently bus-bound.


## Human note + Momentum

If the first time you saw the red storm you thought "surely real code doesn't do that," you are feeling exactly what experienced engineers feel when they audit production GPU workloads. Most accidental performance disasters look like that picture — thousands of tiny expensive crossings that no one intended.

You now have the pictures.

Next section (GPUs for Data Science) we will use exactly this visual discipline on training loops, batch sizes, and mixed precision. You will draw (and mutate) the memory pressure and throughput pictures for real model training steps.

You are ready. The GPU is no longer a mysterious black box — you have sketches of what good and bad look like inside it.
